# 04 - Conformal Prediction Calibration

Calibrate the AdaptiveConformalDetector for anomaly detection with
statistical false alarm rate guarantees.

The conformal prediction layer provides distribution-free anomaly
detection: under exchangeability, the probability of a false alarm
at any single timestep is at most alpha.

## Setup

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from aeroconform.config import AeroConformConfig

config = AeroConformConfig()
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Conformal config: alpha={config.alpha}, cal_window={config.cal_window}, "
      f"adapt_lr={config.adapt_lr}")

## Initialize Foundation Model and Generate Predictions

Run the foundation model on clean synthetic trajectories to obtain
Gaussian mixture predictions for calibration.

In [ ]:
from aeroconform.models.trajectory_model import TrajectoryTransformer

model = TrajectoryTransformer.from_config(config).to(device)
model.eval()

rng = np.random.default_rng(42)


def generate_synthetic_trajectory(n_steps: int, rng: np.random.Generator) -> np.ndarray:
    """Generate a single realistic synthetic ADS-B trajectory."""
    lat = rng.uniform(config.bbox[0], config.bbox[1])
    lon = rng.uniform(config.bbox[2], config.bbox[3])
    alt = rng.uniform(15000, 40000)
    vel = rng.uniform(200, 500)
    hdg = rng.uniform(0, 360)
    vrate = 0.0
    traj = np.zeros((n_steps, 6))
    for t in range(n_steps):
        traj[t] = [lat, lon, alt, vel, hdg, vrate]
        lat += vel * np.cos(np.radians(hdg)) / 3600 / 60
        lon += vel * np.sin(np.radians(hdg)) / 3600 / 60 / np.cos(np.radians(lat))
        alt += vrate / 60
        vel = np.clip(vel + rng.normal(0, 1), 150, 600)
        hdg = (hdg + rng.normal(0, 0.3)) % 360
        vrate = rng.normal(0, 50)
    return traj


# Generate clean calibration data
n_cal_trajectories = 30
clean_trajectories = [generate_synthetic_trajectory(200, rng) for _ in range(n_cal_trajectories)]
print(f"Generated {n_cal_trajectories} clean calibration trajectories")

## Compute Non-Conformity Scores on Clean Data

Run each clean trajectory through the model and compute non-conformity
scores (negative log-likelihood under the predicted distribution).

In [ ]:
from aeroconform.data.preprocessing import delta_encode, compute_norm_stats, normalize
from aeroconform.models.conformal import AdaptiveConformalDetector

# Preprocess trajectories
delta_trajs = [delta_encode(t) for t in clean_trajectories]
norm_stats = compute_norm_stats(delta_trajs)
normed_trajs = [normalize(d, norm_stats) for d in delta_trajs]

# Compute non-conformity scores
detector = AdaptiveConformalDetector.from_config(config)
all_scores = []

for traj_normed in normed_trajs:
    # Pad or truncate to seq_len
    seq_len = config.seq_len
    if len(traj_normed) >= seq_len:
        padded = traj_normed[:seq_len]
    else:
        padded = np.zeros((seq_len, 6))
        padded[:len(traj_normed)] = traj_normed

    x = torch.from_numpy(padded).float().unsqueeze(0).to(device)

    with torch.no_grad():
        means, log_vars, log_weights, _ = model(x)

    # Compute score for each patch
    n_patches = means.shape[1]
    for p in range(n_patches - 1):
        target_start = (p + 1) * config.patch_len
        target_end = target_start + config.patch_len
        if target_end > len(traj_normed):
            break
        target_patch = padded[target_start:target_end].flatten()

        score = detector.compute_nonconformity_score(
            target_patch,
            means[0, p].cpu().numpy(),
            log_vars[0, p].cpu().numpy(),
            log_weights[0, p].cpu().numpy(),
        )
        all_scores.append(score)

all_scores = np.array(all_scores)
print(f"Computed {len(all_scores)} non-conformity scores")
print(f"Score range: [{all_scores.min():.2f}, {all_scores.max():.2f}]")
print(f"Score mean: {all_scores.mean():.2f}, std: {all_scores.std():.2f}")

## Visualize Score Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
ax.hist(all_scores, bins=50, alpha=0.7, color="steelblue", edgecolor="white")
ax.axvline(np.percentile(all_scores, 99), color="red", linestyle="--",
           label=f"99th percentile ({np.percentile(all_scores, 99):.2f})")
ax.set_xlabel("Non-conformity Score")
ax.set_ylabel("Count")
ax.set_title("Clean Data Score Distribution")
ax.legend()
ax.grid(True, alpha=0.3)

# Q-Q plot
ax = axes[1]
sorted_scores = np.sort(all_scores)
theoretical = np.linspace(0, 1, len(sorted_scores))
ax.plot(theoretical, sorted_scores, "o", markersize=2, alpha=0.5)
ax.set_xlabel("Theoretical Quantile")
ax.set_ylabel("Score Value")
ax.set_title("Quantile Plot of Clean Scores")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Calibrate the Conformal Detector

Initialize the calibration set with clean scores. The threshold is set
so that FAR <= alpha under the exchangeability assumption.

In [ ]:
# Use a subset of scores for calibration
cal_scores = all_scores[:config.cal_window]
detector.calibrate(cal_scores)

threshold = detector.get_threshold()
print(f"Calibration complete:")
print(f"  Alpha (target FAR): {config.alpha}")
print(f"  Calibration scores: {len(cal_scores)}")
print(f"  Detection threshold: {threshold:.4f}")
print(f"  Expected clean scores above threshold: {(cal_scores > threshold).mean():.4f}")

## Test on Clean Holdout Data

Verify the false alarm rate on a separate set of clean scores.

In [ ]:
holdout_scores = all_scores[config.cal_window:]
if len(holdout_scores) < 10:
    holdout_scores = all_scores  # Use all if not enough holdout

n_false_alarms = (holdout_scores > threshold).sum()
empirical_far = n_false_alarms / len(holdout_scores)

print(f"Holdout evaluation:")
print(f"  Holdout scores: {len(holdout_scores)}")
print(f"  False alarms: {n_false_alarms}")
print(f"  Empirical FAR: {empirical_far:.4f}")
print(f"  Target alpha:  {config.alpha}")
print(f"  FAR <= alpha + 0.01: {empirical_far <= config.alpha + 0.01}")

## Coverage Guarantee Test

Test the coverage guarantee by running multiple trials and verifying
that the empirical FAR stays within the bound in 95%+ of runs.

In [ ]:
n_trials = 100
trial_fars = []
trial_rng = np.random.default_rng(123)

for trial in range(n_trials):
    # Resample calibration and test scores with random splits
    perm = trial_rng.permutation(len(all_scores))
    split = len(all_scores) // 2
    cal_split = all_scores[perm[:split]]
    test_split = all_scores[perm[split:]]

    trial_detector = AdaptiveConformalDetector(
        alpha=config.alpha,
        cal_window=config.cal_window,
    )
    trial_detector.calibrate(cal_split[:config.cal_window])
    trial_threshold = trial_detector.get_threshold()

    far = (test_split > trial_threshold).mean()
    trial_fars.append(far)

trial_fars = np.array(trial_fars)
coverage_rate = (trial_fars <= config.alpha + 0.01).mean()

print(f"Coverage guarantee test ({n_trials} trials):")
print(f"  Mean empirical FAR: {trial_fars.mean():.4f}")
print(f"  Std empirical FAR:  {trial_fars.std():.4f}")
print(f"  Max empirical FAR:  {trial_fars.max():.4f}")
print(f"  FAR <= alpha+0.01 in {coverage_rate:.1%} of trials")
print(f"  Target: >= 95%")
print(f"  PASS: {coverage_rate >= 0.95}")

## Plot Coverage Results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# FAR distribution across trials
ax = axes[0]
ax.hist(trial_fars, bins=30, alpha=0.7, color="steelblue", edgecolor="white")
ax.axvline(config.alpha, color="red", linestyle="--", linewidth=2,
           label=f"alpha={config.alpha}")
ax.axvline(config.alpha + 0.01, color="orange", linestyle="--", linewidth=2,
           label=f"alpha+0.01={config.alpha + 0.01}")
ax.set_xlabel("Empirical FAR")
ax.set_ylabel("Trial Count")
ax.set_title(f"FAR Distribution ({n_trials} trials)")
ax.legend()
ax.grid(True, alpha=0.3)

# Coverage across alpha values
ax = axes[1]
alpha_values = np.linspace(0.005, 0.1, 20)
empirical_fars_per_alpha = []

for alpha_test in alpha_values:
    trial_det = AdaptiveConformalDetector(alpha=alpha_test, cal_window=config.cal_window)
    trial_det.calibrate(cal_scores)
    t = trial_det.get_threshold()
    far_at_alpha = (holdout_scores > t).mean()
    empirical_fars_per_alpha.append(far_at_alpha)

empirical_fars_per_alpha = np.array(empirical_fars_per_alpha)
ax.plot(alpha_values, empirical_fars_per_alpha, "o-", label="Empirical FAR", color="steelblue")
ax.plot([0, 0.1], [0, 0.1], "k--", alpha=0.3, label="Ideal (FAR = alpha)")
ax.set_xlabel("Nominal alpha")
ax.set_ylabel("Empirical FAR")
ax.set_title("Conformal Coverage Guarantee")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Adaptive Threshold Update

Demonstrate how the threshold adapts over time as new observations
are added to the calibration window.

In [ ]:
# Reset detector and calibrate
adapt_detector = AdaptiveConformalDetector.from_config(config)
adapt_detector.calibrate(cal_scores)

thresholds = []
window_sizes = []

# Stream in new scores and track threshold
stream_scores = all_scores.copy()
trial_rng.shuffle(stream_scores)

for i, score in enumerate(stream_scores[:200]):
    thresholds.append(adapt_detector.get_threshold())
    window_sizes.append(len(adapt_detector.calibration_scores))
    adapt_detector.update(score, is_normal=True)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax = axes[0]
ax.plot(thresholds, color="steelblue", linewidth=1.5)
ax.set_ylabel("Detection Threshold")
ax.set_title("Adaptive Threshold Over Time")
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(stream_scores[:200], color="gray", alpha=0.5, linewidth=0.8, label="Incoming scores")
ax.plot(thresholds, color="red", linewidth=1.5, label="Threshold")
ax.set_xlabel("Timestep")
ax.set_ylabel("Score")
ax.set_title("Scores vs Adaptive Threshold")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Full Predict Demo

Use the `predict` method to classify individual observations with
conformal p-values.

In [ ]:
# Reset detector
demo_detector = AdaptiveConformalDetector.from_config(config)
demo_detector.calibrate(cal_scores)

# Get model predictions for a clean trajectory
traj_normed = normed_trajs[0]
padded = np.zeros((config.seq_len, 6))
padded[:len(traj_normed)] = traj_normed[:config.seq_len]

x = torch.from_numpy(padded).float().unsqueeze(0).to(device)
with torch.no_grad():
    means, log_vars, log_weights, _ = model(x)

# Run predict on several patches
print("Conformal prediction results (clean trajectory):")
print(f"{'Patch':>6} {'Score':>10} {'Threshold':>10} {'P-value':>10} {'Anomaly':>8}")
print("-" * 50)

for p in range(min(10, means.shape[1] - 1)):
    target_start = (p + 1) * config.patch_len
    target_end = target_start + config.patch_len
    if target_end > len(traj_normed):
        break
    target_patch = padded[target_start:target_end].flatten()

    result = demo_detector.predict(
        target_patch,
        means[0, p].cpu().numpy(),
        log_vars[0, p].cpu().numpy(),
        log_weights[0, p].cpu().numpy(),
    )
    print(f"{p:>6} {result['score']:>10.4f} {result['threshold']:>10.4f} "
          f"{result['p_value']:>10.4f} {str(result['is_anomaly']):>8}")

## Summary

This notebook demonstrated:

1. **Non-conformity scores** from the Gaussian mixture model
2. **Calibration** on clean data to set the detection threshold
3. **Coverage guarantee** verification: FAR <= alpha + 0.01 in 95%+ of trials
4. **Adaptive threshold** that evolves with streaming data
5. **Conformal p-values** for individual observations

Next: `05_evaluation.ipynb` benchmarks AeroConform against baselines.